# 📘 Notebook 02: Worlds, States and Rewards

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module A: Foundations · Notebook 2 of 4 in this module · (2 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what a **state** is, and why actions now matter for what happens later
- Build a small game as an **environment** with the `reset` / `step` / `render` interface
- Write the agent-environment loop and record a **video** of an episode
- Compute the **return** of an episode, with and without **discounting**
- Define a **policy** and estimate how good it is by playing many episodes
- State what a **Markov decision process** is, and what the Markov property means
- Use ready-made environments from the **Gymnasium** library

> **Prerequisites:** Notebook 01. Python classes (*From Python to Deep Learning & Fine-Tuning*, Notebook 04) and NumPy arrays.
>
> 🔭 **Why this notebook matters later:** every game in this course, whether we write it ourselves (Snake, Notebook 06) or take it from a library (Lunar Lander, Notebook 09), speaks the interface introduced here. Learn it once and every later notebook starts on familiar ground.

> 📦 **Libraries used in this notebook.**
>
> - **Gymnasium** is the standard collection of reinforcement learning environments, maintained by the Farama Foundation. It began as OpenAI Gym. Its main contribution is a common interface, so that the same agent code can play very different games.
> - **imageio** writes a list of pictures into a video file, which we then show inside the notebook.
>
> The cell below installs what is needed. In Google Colab it takes a few seconds.

In [ ]:
%pip install -q "gymnasium[classic-control,toy-text]>=1.3" imageio imageio-ffmpeg

In [ ]:
import base64
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
from IPython.display import HTML, display

rng = np.random.default_rng(0)


def show_video(frames, fps=4, width=320):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. From one situation to many

### Intuition first
In Notebook 01 every pull of a slot machine was a fresh start. Most of life is not like that. In chess, a move changes the board that you will face on your next turn. In a maze, a step changes where you are standing. An action now has **two** effects: an immediate reward, and a new situation.

That second effect creates a new difficulty. A move can be poor right now and excellent in the long run (sacrificing a piece to win the game), or pleasant now and disastrous later. The agent has to learn to judge actions by everything that follows from them.

### Formal definition
- A **state** $s$ is the situation the agent is in: everything it needs to know to decide.
- At each **time step** $t$ the agent observes the state $S_t$, chooses an action $A_t$, and receives a reward $R_{t+1}$ and the next state $S_{t+1}$.
- An **episode** is one complete game, from a starting state to a **terminal** state where the game ends.

```
          action A_t
   agent ────────────────▶ environment
     ▲                          │
     └──────────────────────────┘
        reward R_t+1, state S_t+1
```

## 2. A world of our own

To see every moving part we write a small game ourselves: a robot in a room, a treasure to reach, two pits to avoid.

```
S . . . .        S  start            #  wall (cannot be entered)
. # . X .        G  treasure, +10    X  pit, -10, the game ends
. # . . .        every step costs -1
. . . X .
. . . . G
```

The class below follows the conventions of the Gymnasium library, which we meet properly in Section 6. An environment needs three methods:

| Method | What it does | What it returns |
|---|---|---|
| `reset()` | starts a new episode | the first state, and a dictionary of extra information |
| `step(action)` | applies one action | next state, reward, `terminated`, `truncated`, extra information |
| `render()` | draws the current situation | a picture as an array of shape (height, width, 3) |

In [ ]:
class GridWorld:
    """A robot in a room. States are numbered row by row: state = row * 5 + column."""

    LAYOUT = ["S....",
              ".#.X.",
              ".#...",
              "...X.",
              "....G"]
    MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]          # 0 up, 1 right, 2 down, 3 left
    ACTION_NAMES = ["up", "right", "down", "left"]
    COLOURS = {"S": (235, 235, 235), ".": (235, 235, 235), "#": (60, 60, 60),
               "X": (200, 60, 60), "G": (240, 200, 40)}

    def __init__(self, slip=0.0, seed=0):
        self.rows, self.cols = len(self.LAYOUT), len(self.LAYOUT[0])
        self.n_states, self.n_actions = self.rows * self.cols, 4
        self.slip = slip                                 # probability that the robot slides sideways
        self.rng = np.random.default_rng(seed)

    def reset(self):
        self.position = (0, 0)
        return self._state(), {}

    def _state(self):
        return self.position[0] * self.cols + self.position[1]

    def step(self, action):
        if self.rng.random() < self.slip:                # a slippery floor: a random move instead
            action = int(self.rng.integers(4))
        row = self.position[0] + self.MOVES[action][0]
        col = self.position[1] + self.MOVES[action][1]
        inside = 0 <= row < self.rows and 0 <= col < self.cols
        if inside and self.LAYOUT[row][col] != "#":      # walls and edges block the move
            self.position = (row, col)
        cell = self.LAYOUT[self.position[0]][self.position[1]]
        reward = {"G": 10.0, "X": -10.0}.get(cell, -1.0)
        terminated = cell in "GX"
        return self._state(), reward, terminated, False, {}

    def render(self, size=48):
        image = np.zeros((self.rows * size, self.cols * size, 3), dtype=np.uint8)
        for r in range(self.rows):
            for c in range(self.cols):
                image[r * size:(r + 1) * size, c * size:(c + 1) * size] = self.COLOURS[self.LAYOUT[r][c]]
                image[r * size, c * size:(c + 1) * size] = 150          # thin grid lines
                image[r * size:(r + 1) * size, c * size] = 150
        r, c = self.position
        image[r * size + 10:(r + 1) * size - 10, c * size + 10:(c + 1) * size - 10] = (40, 100, 220)
        return image


env = GridWorld()
state, info = env.reset()
print("start state:", state)

for action in (1, 1, 2):                                 # right, right, down
    state, reward, terminated, truncated, info = env.step(action)
    print(f"{GridWorld.ACTION_NAMES[action]:>5} -> state {state:>2}, reward {reward:5.1f}, terminated {terminated}")

plt.figure(figsize=(3, 3))
plt.imshow(env.render())
plt.axis("off")
plt.show()

The blue square is the robot. Two details of `step` deserve a second look:

- The **reward is defined by us**, the designers. It says *what* we want (reach the treasure, quickly, without falling in a pit). It does not say *how*.
- `terminated` becomes `True` when the game is over. After that, the only valid call is `reset()`.

## 3. The interaction loop

Every reinforcement learning program, however sophisticated, is built around the same loop: look at the state, choose an action, step, repeat until the episode ends. We write it once as a function that also collects pictures for a video.

In [ ]:
def run_episode(env, policy, max_steps=100, record=False):
    """Play one episode with the given policy. Returns the rewards and the recorded frames."""
    state, info = env.reset()
    rewards, frames = [], []
    for t in range(max_steps):
        if record:
            frames.append(env.render())
        action = policy(state)
        state, reward, terminated, truncated, info = env.step(action)
        rewards.append(reward)
        if terminated or truncated:
            break
    if record:
        frames.append(env.render())
    return rewards, frames


def random_policy(state):
    return int(rng.integers(4))


rewards, frames = run_episode(GridWorld(), random_policy, record=True)
print(f"The episode lasted {len(rewards)} steps and collected a total reward of {sum(rewards):.0f}")
show_video(frames)

Run the cell a few times. The random robot bumps into walls, wanders, and usually ends in a pit or runs out of steps. That is our starting point: an agent that knows nothing.

## 4. How good was that episode? The return

### Intuition first
The agent's goal is not the next reward but the total over the whole episode. Should a reward far in the future count as much as one right now? People usually prefer 100 euros today to 100 euros in ten years, and with good reason: the future is uncertain. Reinforcement learning uses the same idea.

### Formal definition
The **return** $G_t$ is the sum of the rewards from time $t$ onwards, each multiplied by a power of the **discount factor** $\gamma$ (gamma), a number between 0 and 1:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots$$

| $\gamma$ | The agent is | Effect |
|---|---|---|
| 0 | short-sighted | only the next reward counts |
| 0.9 | far-sighted | a reward 10 steps away counts for about a third |
| 1 | patient without limit | all rewards count equally (only safe when episodes always end) |

### Why it matters
From this point on, "a good action" means "an action followed by a high **return**", not a high immediate reward.

In [ ]:
def discounted_return(rewards, gamma):
    total = 0.0
    for reward in reversed(rewards):          # work backwards: G = R + gamma * (the return after it)
        total = reward + gamma * total
    return total


rewards = [-1, -1, -1, 10]                    # three steps, then the treasure
for gamma in (1.0, 0.9, 0.5, 0.0):
    print(f"gamma = {gamma}: return = {discounted_return(rewards, gamma):6.2f}")

The loop runs backwards and uses a fact that will carry the next three notebooks: **the return from now equals the next reward plus gamma times the return from the next step**, $G_t = R_{t+1} + \gamma\,G_{t+1}$.

## 5. Policies

### Formal definition
A **policy** $\pi$ (pi) is the agent's rule of behaviour: it says which action to take in each state. It can be a table, a function, or later a neural network. A policy may also be random: $\pi(a \mid s)$ is then the probability of choosing action $a$ in state $s$.

Learning, in this course, always means **improving a policy**. To improve one we must first be able to say how good it is, and the direct way is to play many episodes and average the returns.

In [ ]:
def right_then_down(state):
    """A hand-written policy: walk along the top row, then down the right-hand column."""
    row, col = divmod(state, 5)
    return 1 if col < 4 else 2


def evaluate(env, policy, episodes=500, gamma=1.0):
    returns = [discounted_return(run_episode(env, policy)[0], gamma) for i in range(episodes)]
    return float(np.mean(returns))


env = GridWorld()
print(f"random policy:          average return {evaluate(env, random_policy):7.2f}")
print(f"right, then down:       average return {evaluate(env, right_then_down):7.2f}")

rewards, frames = run_episode(env, right_then_down, record=True)
show_video(frames)

The hand-written policy reaches the treasure in 8 steps: seven steps at −1 and a final +10 give a return of 3. No policy can do better in this room, because the treasure is 8 moves away. In the next notebooks the agent will have to find such a policy without being told.

## 6. The formal picture: Markov decision processes

### Formal definition
A **Markov decision process (MDP)** is the mathematical description of an environment. It consists of:

| Part | Symbol | In our room |
|---|---|---|
| a set of states | $\mathcal{S}$ | the 25 cells |
| a set of actions | $\mathcal{A}$ | up, right, down, left |
| transition probabilities | $p(s' \mid s, a)$ | where each move leads |
| a reward function | $r(s, a, s')$ | −1, +10 or −10 |
| a discount factor | $\gamma$ | our choice |

The name comes from the **Markov property**: the next state and reward depend only on the **current** state and action, not on how the agent got there. The state must therefore contain everything relevant. A single photograph of a moving ball is not a Markov state, because it does not show which way the ball is going. Notebook 11 returns to exactly this problem.

### Environments with chance
So far the room has been **deterministic**: the same action in the same state always has the same result. Real environments rarely are. Our class has a `slip` parameter: with that probability the robot's move is replaced by a random one, as on a wet floor.

In [ ]:
for slip in (0.0, 0.2, 0.4):
    env = GridWorld(slip=slip, seed=1)
    print(f"slip = {slip}: 'right, then down' earns an average return of {evaluate(env, right_then_down):6.2f}")

The same policy, which was perfect on a dry floor, becomes poor on a slippery one: its route passes right next to a pit. In a world with chance, the best policy is the one with the best return **on average**, and it may well take a longer, safer road.

> ⚠️ **Common pitfalls**
>
> - **Confusing reward and return.** The reward is one number for one step. The return is the discounted sum of everything that follows.
> - **Putting the solution into the reward.** Reward the outcome you want (treasure reached), not the route you imagine (go right). Notebook 18 shows agents exploiting badly designed rewards.
> - **Calling `step` after the episode has ended.** Always `reset()` first.

## 7. Gymnasium: environments from a library

We wrote `GridWorld` in the Gymnasium style on purpose. The library offers dozens of ready-made environments with the same three methods, so the loop we wrote runs on them unchanged.

**Frozen Lake** is a library version of our room: cross a frozen lake from the start to a present without falling into a hole.

In [ ]:
lake = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")

print("observation space:", lake.observation_space)      # 16 numbered squares
print("action space:     ", lake.action_space)           # 0 left, 1 down, 2 right, 3 up

state, info = lake.reset(seed=0)
frames = [lake.render()]
terminated = truncated = False
while not (terminated or truncated):
    action = lake.action_space.sample()                   # a random action
    state, reward, terminated, truncated, info = lake.step(action)
    frames.append(lake.render())

print(f"finished in state {state} with final reward {reward}")
show_video(frames)

Three things are new:

- **Spaces.** `observation_space` and `action_space` describe what states and actions look like. `Discrete(16)` means "an integer from 0 to 15".
- **`render_mode="rgb_array"`** asks for pictures as arrays, which is what our video function needs.
- **`truncated`.** Gymnasium ends an episode in two ways. `terminated` means the game itself is over (hole or present). `truncated` means a time limit cut it short. The difference will matter when we compute returns in Notebook 05.

The interface does not care what kind of game is behind it. **CartPole** is a physics simulation: a pole is hinged to a cart, and the agent pushes the cart left or right to keep the pole from falling.

In [ ]:
cartpole = gym.make("CartPole-v1", render_mode="rgb_array")
print("observation space:", cartpole.observation_space)
print("action space:     ", cartpole.action_space)

state, info = cartpole.reset(seed=0)
print("first state:", np.round(state, 3))

frames, total = [cartpole.render()], 0.0
terminated = truncated = False
while not (terminated or truncated):
    state, reward, terminated, truncated, info = cartpole.step(cartpole.action_space.sample())
    total += reward
    frames.append(cartpole.render())

print(f"the pole stayed up for {total:.0f} steps")
show_video(frames, fps=30, width=420)

Here the state is not a square number but **four real numbers**: the cart's position and speed, and the pole's angle and how fast it is turning. There are infinitely many such states. A table with one row per state is impossible, and that problem will open Module C. For now, notice that a random agent keeps the pole up for only a moment. The best possible score in this environment is 500.

---
## ✏️ Exercises

The exercises use `GridWorld`, `run_episode`, `evaluate`, `discounted_return` and `show_video` from above.

### Exercise 1 (A return by hand)
An episode produces the rewards `[0, 0, 5, -2, 10]`. Compute its return for gamma = 0.9 on paper, as $0 + 0.9 \cdot 0 + 0.9^2 \cdot 5 + \dots$, then check with `discounted_return`. How does the return change if the reward of 10 arrives one step later (insert a 0 before it)?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
rewards = [0, 0, 5, -2, 10]
by_hand = 0 + 0.9 * 0 + 0.9**2 * 5 + 0.9**3 * -2 + 0.9**4 * 10
print(f"by hand:   {by_hand:.3f}")
print(f"function:  {discounted_return(rewards, 0.9):.3f}")
print(f"one step later: {discounted_return([0, 0, 5, -2, 0, 10], 0.9):.3f}")
```

*Delaying the 10 by one step multiplies its contribution by 0.9. Discounting makes the agent prefer to be quick, without any explicit penalty per step.*
</details>

### Exercise 2 (A safer route)
Write a policy `down_then_right` that walks down the left-hand column and then along the bottom row. Evaluate it and `right_then_down` on a slippery floor (`slip=0.2`, `seed=1`). Which one is better there, and why?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def down_then_right(state):
    row, col = divmod(state, 5)
    return 2 if row < 4 else 1

env = GridWorld(slip=0.2, seed=1)
print(f"right, then down: {evaluate(env, right_then_down):6.2f}")
env = GridWorld(slip=0.2, seed=1)
print(f"down, then right: {evaluate(env, down_then_right):6.2f}")
```

*Both routes have 8 steps, but the second one never passes next to a pit until the very end, and for most of its length a slip costs only a step or two. On a dry floor the two policies are equally good. Once chance enters, they are not.*
</details>

### Exercise 3 (How long does a random robot survive?)
Play 2,000 episodes with `random_policy` on the dry floor. Print the average episode length, and the percentage of episodes that ended at the treasure, in a pit, or by running out of the 100 steps. Hint: the last reward of an episode tells you how it ended.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
env = GridWorld()
lengths, endings = [], {"treasure": 0, "pit": 0, "out of time": 0}
for episode in range(2000):
    rewards, frames = run_episode(env, random_policy)
    lengths.append(len(rewards))
    if rewards[-1] == 10:
        endings["treasure"] += 1
    elif rewards[-1] == -10:
        endings["pit"] += 1
    else:
        endings["out of time"] += 1

print(f"average length: {np.mean(lengths):.1f} steps")
for ending, count in endings.items():
    print(f"{ending:12s} {100 * count / 2000:5.1f}%")
```
</details>

### Exercise 4 (Inspecting environments)
For each of `"FrozenLake-v1"`, `"CartPole-v1"`, `"MountainCar-v0"` and `"Taxi-v4"`, create the environment and print its observation space and action space. For which of them could you build a table with one row per state?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for name in ("FrozenLake-v1", "CartPole-v1", "MountainCar-v0", "Taxi-v4"):
    environment = gym.make(name)
    print(f"{name:15s} observations: {environment.observation_space}   actions: {environment.action_space}")
    environment.close()
```

*Frozen Lake (16 states) and Taxi (500 states) are `Discrete`, so a table works. CartPole and MountainCar have `Box` observations, vectors of real numbers, and no table can list them all.*
</details>

### Exercise 5 (A coin on the way)
Make a subclass `CoinWorld` of `GridWorld` with a coin in the cell at row 2, column 2. Stepping on that cell gives an extra reward of +5, **once per episode**. Compare the return of `right_then_down` in both worlds, then write a policy that picks up the coin on its way to the treasure. Is the detour worth it?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
class CoinWorld(GridWorld):
    def reset(self):
        self.coin_taken = False
        return super().reset()

    def step(self, action):
        state, reward, terminated, truncated, info = super().step(action)
        if self.position == (2, 2) and not self.coin_taken:
            reward += 5.0
            self.coin_taken = True
        return state, reward, terminated, truncated, info


def via_the_coin(state):
    row, col = divmod(state, 5)
    if row == 0 and col < 2:
        return 1                      # right along the top row to column 2
    if col == 2 and row < 4:
        return 2                      # down through the coin to the bottom row
    return 1                          # right along the bottom row

print(f"right, then down: {evaluate(CoinWorld(), right_then_down):.2f}")
print(f"via the coin:     {evaluate(CoinWorld(), via_the_coin):.2f}")
```

*The route through the coin is also 8 steps long, so the coin is pure profit. One subtle point: with a coin that can be taken only once, the cell number alone is no longer a full Markov state. "Has the coin been taken?" is part of the situation too.*
</details>

### Exercise 6 (Watching chance at work (a little harder))
Record a video of `right_then_down` on a floor with `slip=0.3`. Then estimate, from 2,000 episodes for each value, the probability that this policy reaches the treasure for `slip` = 0, 0.1, 0.2, 0.3, 0.4 and 0.5, and plot it.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
rewards, frames = run_episode(GridWorld(slip=0.3, seed=3), right_then_down, record=True)
show_video(frames)

slips = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
success = []
for slip in slips:
    env = GridWorld(slip=slip, seed=0)
    wins = sum(run_episode(env, right_then_down)[0][-1] == 10 for episode in range(2000))
    success.append(100 * wins / 2000)
    print(f"slip = {slip}: reaches the treasure in {success[-1]:.0f}% of episodes")

plt.figure(figsize=(7, 4))
plt.plot(slips, success, marker="o")
plt.title("A fixed policy in an increasingly slippery world")
plt.xlabel("probability of slipping"); plt.ylabel("% of episodes that reach the treasure")
plt.grid(True)
plt.show()
```
</details>

## 🔑 Key takeaways

- A **state** describes the agent's situation. Actions now change both the reward and the next state.
- An environment offers **`reset`**, **`step`** and **`render`**. The agent-environment **loop** is the same for every game.
- The **return** is the discounted sum of future rewards, and $G_t = R_{t+1} + \gamma\,G_{t+1}$.
- A **policy** maps states to actions. Its quality is the average return it earns.
- A **Markov decision process** is states, actions, transition probabilities, rewards and a discount factor. In a Markov state, the past adds nothing to the present.
- **Gymnasium** provides many environments behind one interface; `terminated` and `truncated` are two different ways for an episode to stop.

---
**Next:** *Notebook 03: Planning with the Bellman Equations*, where we compute the best possible behaviour for a world whose rules we know.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*